# Game networking basics
* every game in this course so far has been strictly single-player - one process, running entirely on one machine, nobody else involved
* a **multiplayer** game needs multiple separate programs (usually: several **clients**, one per player, plus one **server**) running on different machines, talking to each other over a network
* this lesson covers the fundamental networking concepts every multiplayer game is built on - sockets, TCP vs UDP, the client-server model, and why the server (not the client) should be the "boss" of the game state
* lesson 34 then builds a real, working (simulated) multiplayer game server on top of everything we learn here

## client-server vs peer-to-peer
* **peer-to-peer**: every player's game connects directly to every other player's game, with no central authority - simple for 2 players, but gets complicated fast (everyone needs everyone else's address, and nobody is really "in charge" if players disagree about what happened)
* **client-server**: every player's game (the **client**) only talks to ONE central program (the **server**) - the server is the single source of truth, clients just send it their intentions ("I want to move right") and receive updates about the actual game state back
* almost every real multiplayer game uses client-server, for a huge reason: an **authoritative server** - the server decides what ACTUALLY happens (movement, damage, who wins), clients just display it. if a client just directly told everyone else "I did 9999 damage!", a modified/cheating client could say anything it wants - the server checking and enforcing the real rules is what keeps a game fair
* we will build exactly this pattern in this lesson and the next: clients send REQUESTS, the server decides the REAL outcome and sends everyone the RESULT

## sockets: the basic building block
* python's standard library `socket` module gives us direct access to the operating system's networking - no extra install needed
* a **socket** is one endpoint of a network connection, identified by an IP address (which MACHINE) and a port number (which PROGRAM on that machine - a machine can run many networked programs at once, each with its own port)
* we always use `127.0.0.1` (aka "localhost") in this lesson - it means "this same machine", so everything here runs safely offline, exactly like our http.server demos in earlier lessons (recap lesson 14)
* `socket.socket(family, type)` creates one - `type` is the big fork in the road: `socket.SOCK_STREAM` (TCP) or `socket.SOCK_DGRAM` (UDP), covered right below

## TCP vs UDP - the two core transport protocols
* **TCP** (`SOCK_STREAM`) is **reliable** and **ordered**: every byte you send is guaranteed to arrive, in the exact order you sent it (or the connection fails loudly) - the cost is more overhead (acknowledgements, retransmission of lost data) and a bit more latency
* **UDP** (`SOCK_DGRAM`) is **unreliable** and **unordered**: a packet might get lost, duplicated, or arrive out of order, and NOBODY tells you if that happened - the upside is much lower overhead and latency, since there's no waiting-for-acknowledgement machinery at all
* for games, the classic rule of thumb:
  * use **TCP** for anything that MUST arrive and MUST arrive correctly: chat messages, trades, login, "you won the match" - losing or reordering one of these would be a real, visible bug
  * use **UDP** for fast, real-time, constantly-repeating data like player position: if ONE position update packet gets lost, who cares - a newer one is already on its way, a split second later. WAITING for a guaranteed-delivered retransmission of stale position data would actually make the game feel WORSE (more laggy), not better
* many real multiplayer games use BOTH at once: TCP for the "important stuff" channel, UDP for the "fast and frequent" movement channel

## a raw TCP demo: server and client, talking over a real socket
* same reliable pattern this whole course has used since lesson 14: run the "server" in a background thread, then talk to it as the "client" from the very same notebook - this keeps everything self-contained and offline, but it's REAL socket communication, the exact same API you'd use talking to a server on a different machine across the internet
* server side: `bind()` (claim a port), `listen()` (start accepting connections), `accept()` (wait for and accept ONE incoming connection, blocking until a client shows up)
* client side: `connect()` to the server's address+port, then `send()`/`recv()` to exchange bytes

In [1]:
import socket
import threading

def run_echo_server(server_socket):
    """Accept one client connection, echo back whatever it sends, once, then stop."""
    connection, client_address = server_socket.accept()  # blocks here until a client connects
    print(f"server: accepted a connection from {client_address}")
    data = connection.recv(1024)  # blocks until SOME data arrives (up to 1024 bytes at a time)
    print(f"server: received {data!r}")
    connection.sendall(b"echo: " + data)  # sendall() makes sure ALL the bytes actually get sent, not just some of them
    connection.close()

# AF_INET means "regular IPv4 addressing" (ip + port), SOCK_STREAM means TCP
tcp_server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
tcp_server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)  # lets us reuse this port quickly if we rerun this cell
tcp_server_socket.bind(("127.0.0.1", 0))  # port 0 = "operating system, pick any free port for me" (recap lesson 14)
tcp_port = tcp_server_socket.getsockname()[1]  # ask the OS which port it actually gave us
tcp_server_socket.listen(1)  # start accepting connections, with room for 1 waiting connection in the queue

server_thread = threading.Thread(target=run_echo_server, args=(tcp_server_socket,), daemon=True)
server_thread.start()

print(f"tcp echo server listening on 127.0.0.1:{tcp_port}")

tcp echo server listening on 127.0.0.1:35283


In [2]:
import time
time.sleep(0.1)  # tiny pause to make sure the server thread above actually reached accept() before we try to connect

# the "client" side - a completely separate socket object, playing the other role in this same conversation
tcp_client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
tcp_client_socket.connect(("127.0.0.1", tcp_port))  # this is the call that actually triggers the server's accept()

tcp_client_socket.sendall(b"hello from the client!")
response = tcp_client_socket.recv(1024)
print("client: received", response)

tcp_client_socket.close()
server_thread.join(timeout=2)
tcp_server_socket.close()
print("tcp demo finished, both sockets closed")

server: accepted a connection from ('127.0.0.1', 43844)
server: received b'hello from the client!'
client: received b'echo: hello from the client!'
tcp demo finished, both sockets closed


## the message-framing problem
* TCP gives us a **byte stream**, not a list of discrete "messages" - if you call `send()` twice quickly, the other side's `recv()` might get BOTH messages concatenated together in one call, or split weirdly across two calls, there's no guarantee they line up 1-to-1
* this means a real protocol needs SOME way to say "this is where one message ends and the next begins" - two classic, simple solutions:
  * **delimiter-based framing**: pick a byte that never appears inside a message (a newline `\n` works great for JSON-per-line, since JSON text itself never contains a raw newline) and read up to that delimiter each time
  * **length-prefixed framing**: send the LENGTH of the message first (e.g. 4 bytes representing a number), then read exactly that many bytes - used when your data might genuinely contain newlines (raw binary data, for example)
* we use the newline-delimited approach here since we're sending small JSON messages - lesson 34's `asyncio` streams even give us a ready-made `readline()` for this, no manual buffering needed there

In [3]:
import json

def run_framing_demo_server(server_socket):
    connection, _ = server_socket.accept()
    buffer = b""  # we accumulate raw bytes here until we've seen a full line
    messages_received = []
    while len(messages_received) < 2:  # we know the client will send exactly 2 messages in this demo
        chunk = connection.recv(1024)
        buffer += chunk
        while b"\n" in buffer:  # buffer might now contain 0, 1, or even both messages at once!
            line, buffer = buffer.split(b"\n", 1)  # split off the FIRST complete line, keep the rest in buffer
            messages_received.append(json.loads(line))
    connection.close()
    framing_results.append(messages_received)

framing_results = []
framing_server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
framing_server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
framing_server_socket.bind(("127.0.0.1", 0))
framing_port = framing_server_socket.getsockname()[1]
framing_server_socket.listen(1)
framing_thread = threading.Thread(target=run_framing_demo_server, args=(framing_server_socket,), daemon=True)
framing_thread.start()
time.sleep(0.1)

framing_client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
framing_client_socket.connect(("127.0.0.1", framing_port))

# send BOTH messages in one go, glued together - this deliberately simulates the "concatenated in one recv()" scenario
message_one = json.dumps({"type": "move", "direction": "up"}).encode() + b"\n"
message_two = json.dumps({"type": "chat", "text": "hi there"}).encode() + b"\n"
framing_client_socket.sendall(message_one + message_two)  # both messages sent as ONE single send() call

time.sleep(0.2)  # give the server thread a moment to receive+parse everything
framing_client_socket.close()
framing_thread.join(timeout=2)
framing_server_socket.close()

print("messages the server correctly separated back out:", framing_results[0])
# even though we sent both messages glued together in ONE sendall() call, our newline-splitting buffer correctly
# pulled them apart into two separate, correctly-parsed json objects - exactly the problem framing solves

messages the server correctly separated back out: [{'type': 'move', 'direction': 'up'}, {'type': 'chat', 'text': 'hi there'}]


## a raw UDP demo
* UDP has no `listen()`/`accept()`/`connect()` - there's no persistent "connection" at all, just individual packets sent to an address
* `sendto(data, address)` on the sending side, `recvfrom(bufsize)` on the receiving side (which also tells you WHO just sent you that packet)
* notice how much simpler the code is than TCP - that simplicity (no handshake, no connection state to track) is a big part of why UDP has lower overhead

In [4]:
udp_server_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)  # SOCK_DGRAM = UDP
udp_server_socket.bind(("127.0.0.1", 0))
udp_port = udp_server_socket.getsockname()[1]

def run_udp_server(server_socket, results):
    data, sender_address = server_socket.recvfrom(1024)  # blocks until ONE packet arrives, then returns it + who sent it
    results.append((data, sender_address))

udp_results = []
udp_thread = threading.Thread(target=run_udp_server, args=(udp_server_socket, udp_results), daemon=True)
udp_thread.start()
time.sleep(0.1)

udp_client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
# no connect() needed at all - we just fire a packet straight at the destination address
udp_client_socket.sendto(json.dumps({"type": "position", "x": 12, "y": 7}).encode(), ("127.0.0.1", udp_port))

udp_thread.join(timeout=2)
received_data, sender = udp_results[0]
print("udp server received:", json.loads(received_data), "from", sender)

udp_client_socket.close()
udp_server_socket.close()

udp server received: {'type': 'position', 'x': 12, 'y': 7} from ('127.0.0.1', 47388)


## the authoritative server pattern
* lets put "client-server" into practice with a tiny but important example: the CLIENT sends what it WANTS to do, the SERVER decides what ACTUALLY happens, and only the server's decision is treated as the truth
* this reuses our RPG's wall-collision idea from lesson 31 - the client proposes a move, the server checks it against the (server-side!) map before accepting it
* why does this matter so much for real games? if the CLIENT just decided "yep, I moved through that wall, trust me" and told the server/other players that as fact, a modified client could walk through walls, teleport anywhere, or claim any outcome it wants - checking on the server, which the player cannot modify, is what actually prevents that

In [5]:
# a tiny server-side world - the real, authoritative map (the client never gets to modify this copy)
AUTHORITATIVE_MAP = [
    "####",
    "#..#",
    "#.##",
    "####",
]
server_side_player_position = [1, 1]  # server's own record of where the player REALLY is


def handle_move_request(requested_dx, requested_dy):
    """The server decides whether a requested move is actually allowed - this is the authoritative check."""
    proposed_x = server_side_player_position[0] + requested_dx
    proposed_y = server_side_player_position[1] + requested_dy
    if AUTHORITATIVE_MAP[proposed_y][proposed_x] == "#":
        return {"accepted": False, "reason": "that tile is a wall", "position": list(server_side_player_position)}
    server_side_player_position[0] = proposed_x  # only NOW, after the check passed, do we actually update the truth
    server_side_player_position[1] = proposed_y
    return {"accepted": True, "position": list(server_side_player_position)}


# a well-behaved client only ever requests legal moves...
print("client requests move right:", handle_move_request(1, 0))

# ...but lets simulate a CHEATING or just plain BUGGY client trying to walk straight into a wall
print("cheating client requests move up (straight into a wall):", handle_move_request(0, -1))

# the key takeaway: no matter WHAT the client asked for, server_side_player_position only ever changed
# when the server's OWN check allowed it - the client has zero power to just force a different outcome
print("final, real, server-side position:", server_side_player_position)

client requests move right: {'accepted': True, 'position': [2, 1]}
cheating client requests move up (straight into a wall): {'accepted': False, 'reason': 'that tile is a wall', 'position': [2, 1]}
final, real, server-side position: [2, 1]


## a quick word on latency and what's coming in lesson 34
* even on a fast connection, there's always SOME delay (**latency** / "ping", usually measured in milliseconds) between a client sending something and the server's reply arriving back
* for a slow-paced game (turn-based, like our RPG) this barely matters - a few hundred milliseconds of delay before your move confirms is unnoticeable
* for a fast-paced action game, waiting for a server round-trip before your character visibly moves would feel terrible - real games use tricks like **client-side prediction** (show the move immediately, locally, before the server even confirms it) and **server reconciliation** (quietly correct the client's position if the server's authoritative answer ever disagrees) to hide this latency from the player
* lesson 34 builds a real (simulated) multiplayer server and touches on exactly these ideas in practice

## quick comparison to other languages/engines
* the raw `socket` module here maps directly onto the exact same **Berkeley sockets API** used by C, C++, Java (`java.net.Socket`/`DatagramSocket`), C# (`System.Net.Sockets`), Go (`net` package) and virtually every other language - sockets are one of the most universal, OS-level programming interfaces that exists, python's `socket` module is a thin wrapper around the very same system calls
* real game engines rarely make you write raw sockets by hand though: **Unity** has built-in Netcode for GameObjects (plus popular third-party options like Mirror and Photon), **Godot** has a high-level multiplayer API built on top of ENet, and dedicated game-networking LIBRARIES like **ENet** and **RakNet** (both C/C++, used across many engines) specifically implement "reliable-UDP" - UDP's low overhead, but with an OPTIONAL reliability layer bolted on top for the messages that need it, the best of both worlds we discussed above
* browser-based multiplayer games lean heavily on **WebSockets** (a persistent, full-duplex TCP-based connection built for the web) - Node.js's `Socket.IO` library is an extremely popular choice specifically for this, conceptually very similar to the client-server socket pattern we just built, just wrapped in a browser-friendly API

## Exercises
1. modify the TCP echo server so the CLIENT's message gets uppercased before being echoed back, instead of just repeated as-is
2. extend the message-framing demo to correctly split apart THREE glued-together messages sent in a single `sendall()` call, instead of two
3. add a second authoritative-server check to `handle_move_request`-style logic: reject a move that would go outside the map's bounds entirely (not just onto a wall tile) - test it with a move that would step off the edge
4. write a small UDP "position broadcast" simulation: one UDP socket sends 3 different `{"x":.., "y":..}` position packets in a row to the same server socket, and the server prints each one as it arrives (recap `recvfrom()`)

In [6]:
# TODO: solve exercise 1 here (uppercase the echoed message)


# TODO: solve exercise 2 here (correctly split 3 glued-together messages)


# TODO: solve exercise 3 here (reject a move that would go outside the map bounds)


# TODO: solve exercise 4 here (UDP: send 3 position packets, server prints each)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [7]:
# sample solution 1 - uppercase the echoed message
def run_uppercase_server(server_socket):
    connection, _ = server_socket.accept()
    data = connection.recv(1024)
    connection.sendall(data.upper())  # the only real change - .upper() on the raw bytes before sending back
    connection.close()

uppercase_server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
uppercase_server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
uppercase_server_socket.bind(("127.0.0.1", 0))
uppercase_port = uppercase_server_socket.getsockname()[1]
uppercase_server_socket.listen(1)
uppercase_thread = threading.Thread(target=run_uppercase_server, args=(uppercase_server_socket,), daemon=True)
uppercase_thread.start()
time.sleep(0.1)

uppercase_client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
uppercase_client_socket.connect(("127.0.0.1", uppercase_port))
uppercase_client_socket.sendall(b"shout this back at me")
print(uppercase_client_socket.recv(1024))
uppercase_client_socket.close()
uppercase_thread.join(timeout=2)
uppercase_server_socket.close()


# sample solution 2 - split three glued-together messages
def run_triple_framing_server(server_socket, results):
    connection, _ = server_socket.accept()
    buffer = b""
    messages = []
    while len(messages) < 3:
        buffer += connection.recv(1024)
        while b"\n" in buffer:
            line, buffer = buffer.split(b"\n", 1)
            messages.append(json.loads(line))
    connection.close()
    results.append(messages)

triple_results = []
triple_server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
triple_server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
triple_server_socket.bind(("127.0.0.1", 0))
triple_port = triple_server_socket.getsockname()[1]
triple_server_socket.listen(1)
triple_thread = threading.Thread(target=run_triple_framing_server, args=(triple_server_socket, triple_results), daemon=True)
triple_thread.start()
time.sleep(0.1)

triple_client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
triple_client_socket.connect(("127.0.0.1", triple_port))
glued = b"".join(json.dumps({"n": i}).encode() + b"\n" for i in range(3))  # all 3 messages glued into ONE bytes object
triple_client_socket.sendall(glued)
time.sleep(0.2)
triple_client_socket.close()
triple_thread.join(timeout=2)
triple_server_socket.close()
print("split back out into 3 messages:", triple_results[0])


# sample solution 3 - reject a move outside the map bounds
def handle_move_request_with_bounds_check(dx, dy, position, world_map):
    proposed_x, proposed_y = position[0] + dx, position[1] + dy
    if not (0 <= proposed_y < len(world_map)) or not (0 <= proposed_x < len(world_map[0])):
        return {"accepted": False, "reason": "that would go outside the map entirely", "position": list(position)}
    if world_map[proposed_y][proposed_x] == "#":
        return {"accepted": False, "reason": "that tile is a wall", "position": list(position)}
    position[0], position[1] = proposed_x, proposed_y
    return {"accepted": True, "position": list(position)}

bounds_test_position = [1, 1]
print(handle_move_request_with_bounds_check(-5, 0, bounds_test_position, AUTHORITATIVE_MAP))  # way off the left edge


# sample solution 4 - UDP position broadcast simulation
def run_position_listener(server_socket, count):
    for _ in range(count):
        data, sender = server_socket.recvfrom(1024)
        print("server received position update:", json.loads(data), "from", sender)

position_server_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
position_server_socket.bind(("127.0.0.1", 0))
position_port = position_server_socket.getsockname()[1]
position_thread = threading.Thread(target=run_position_listener, args=(position_server_socket, 3), daemon=True)
position_thread.start()
time.sleep(0.1)

position_client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
for x, y in [(1, 1), (2, 1), (2, 2)]:
    position_client_socket.sendto(json.dumps({"x": x, "y": y}).encode(), ("127.0.0.1", position_port))
    time.sleep(0.05)  # small gap so the 3 packets dont all arrive in one instant burst

position_thread.join(timeout=2)
position_client_socket.close()
position_server_socket.close()

b'SHOUT THIS BACK AT ME'


split back out into 3 messages: [{'n': 0}, {'n': 1}, {'n': 2}]
{'accepted': False, 'reason': 'that would go outside the map entirely', 'position': [1, 1]}
server received position update: {'x': 1, 'y': 1} from ('127.0.0.1', 36973)
server received position update: {'x': 2, 'y': 1} from ('127.0.0.1', 36973)


server received position update: {'x': 2, 'y': 2} from ('127.0.0.1', 36973)


* congratulation you finished this lesson

## what we learned
* why real multiplayer games use a client-server architecture with an authoritative server, instead of peer-to-peer
* sockets: `socket.socket()`, `bind()`/`listen()`/`accept()`/`connect()`/`send()`/`recv()` for TCP, `sendto()`/`recvfrom()` for UDP
* TCP (reliable, ordered) vs UDP (unreliable, unordered, lower overhead) and which one games typically use for what kind of data
* the message-framing problem - TCP is a byte stream, not a list of messages - and a simple newline-delimited JSON solution
* the authoritative server pattern: the client proposes, the server decides, and only the server's decision is ever treated as the real game state
* latency/lag and a first mention of client-side prediction and server reconciliation, which lesson 34 explores properly